# Molecular Properties and Library Selection.

## Background
This practical introduces molecular descriptors, chemical space, molecular similarity, hierarchical clustering and diversity selection in the context of drug discovery.

## Learning Objectives
- Work with molecular structures using RDKit
- Calculate molecular descriptors
- Explore chemical space using PCA
- Compute molecular similarity
- Perform hierarchical clustering
- Select representative and diverse compounds

## Part 1. Introduction to Data Analysis with Python

For this tutorial, we will use the Fisher’s Iris data set to exemplify how python can be used to extract relevant information from a multivariate data set. This data set contains information about four features (the length and the width of the sepals and petals) obtained by 50 samples from each of three species of Iris (i.e. iris setosa, iris virginica and iris versicolor). Note that molecular descriptors would be considered as molecular features, so we will give them the same treatment as these four flower‐related features.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from io import StringIO

In [ ]:
iris=load_iris(as_frame=True).frame
iris.head()

In [ ]:
iris.describe()

In [ ]:
sns.pairplot(iris,hue='target')

### EXERCISE 
The following commands illustrate fundamental data manipulation and visualization operations in Python. Execute each command and briefly explain its purpose and the output it produces.

| Python command                                                     | Description |
| ------------------------------------------------------------------ | ----------- |
| `vector = [6, 9, 0]`                                               |             |
| `vector[1]`                                                        |             |
| `from sklearn.datasets import load_iris`                           |             |
| `iris = load_iris(as_frame=True).frame`                            |             |
| `iris.head()`                                                      |             |
| `iris.describe()`                                                  |             |
| `iris["sepal length (cm)"].hist()`                                 |             |
| `iris["sepal width (cm)"].hist()`                                  |             |
| `dist_matrix = squareform(pdist(iris.iloc[:,0:4]))`                |             |
| `sns.heatmap(dist_matrix)`                                         |             |
| `plt.scatter(iris["petal width (cm)"], iris["petal length (cm)"])` |             |


In [ ]:
# Run your code here:

## Part 2.  Calculating molecular descriptors

Our research group is interested in studying the molecular diversity of neuraminidase and
p38 MAP kinase inhibitors and to evaluate the common characteristics that dual inhibitors may
have.
Information about active compounds against these targets are available in literature. In this
exercise we will use the Stahl data set [J. Med.Chem., 2001, 44(7), 1035‐1042.] which can be downloaded from Cheminformatics web
portal (http://cheminformatics.org/datasets/).

<div class="alert alert-block alert-warning">
<strong>Task</strong><br>

Download the two databases containing the molecular structure of neuraminidase and p38 MAP kinase
inhibitors in SMILE format.
<br>
It is recommended to save these files into a new folder named DATABASES.
(stahl-neuraminidase.smi and stahl-p38MAPkinase.smi).
</div>

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw,Descriptors,rdMolDescriptors,AllChem
from rdkit import DataStructs
from rdkit import RDLogger

RDLogger.DisableLog('rdApp.*')

In [ ]:
'''stahl_df=pd.read_csv('DATABASES/stahl_dataset.csv', sep=',')
stahl_df.head()
stahl_df['mol']=stahl_df['smiles'].apply(Chem.MolFromSmiles)
stahl_df=stahl_df[stahl_df.mol.notnull()].reset_index(drop=True)
'''

In [3]:
csv_data = """
smiles,target
CC(=O)NC1C(OC(=CC1[NH+]=C(N)N)C(=O)[O-])C(=O)C(O)CO,neuraminidase
CCC(CC)C(=O)Nc1cc(ccc1NC(=O)C)C(=O)[O-],neuraminidase
NC(N)[NH+]=C1C=C(C=CC1N2C(=O)CCC2(CO)CO)C(=O)[O-],neuraminidase
CCC(CC)Nc1cc(ccc1N2C(=O)CCC2(CO)CO)C(=O)[O-],neuraminidase
CCCN(CCc1ccccc1)C(=O)C2OC(=CC([NH3+])C2NC(=O)C)C(=O)[O-],neuraminidase
CC(=O)NC1C=CC(=CC1=[NH+]C(N)N)C(=O)[O-],neuraminidase
CC(=O)Nc1ccc(cc1NC(=O)CO)C(=O)[O-],neuraminidase
CC(=O)Nc1ccc(cc1NC(=O)C[NH3+])C(=O)[O-],neuraminidase
CC(=O)NC1C(O)CC(OC1C(O)C(O)CO)P(=O)(O)O,neuraminidase
CC(=O)Nc1c(O)cc(cc1N(=O)=O)C(=O)[O-],neuraminidase
CC(=O)Nc1c(N)cc(cc1O)C(=O)[O-],neuraminidase
CC(=O)Nc1ccc(cc1N)C(=O)[O-],neuraminidase
CC(=O)NC1C(OC(=CC1[NH+]=C(N)N)C(=O)[O-])C(O)C(O)CO,neuraminidase
CC(=O)NC1C(O)C=C(OC1C(O)C(O)CO)C(=O)[O-],neuraminidase
CC(=O)NC1C(O)CC(O)(OC1C(O)C(O)CO)C(=O)[O-],neuraminidase
CCCN(C)C(=O)C1OC(=CC([NH+]=C(N)N)C1NC(=O)C)C(=O)[O-],neuraminidase
CCN(CC)C(=O)C1OC(=CC([NH3+])C1NC(=O)C)C(=O)[O-],neuraminidase
CCS(=O)c1ccc(cc1)c2nc(c([nH]2)c3ccncc3)c4ccc(F)cc4,MAPK
CCC(CC)(CC)c1nc(c([nH]1)c2ccncc2)c3ccccc3,MAPK
COc1ccc(cc1)c2nc([nH]c2c3ccncc3)c4ccc(Cl)cc4,MAPK
Fc1ccc(cc1)c2nc3S(=O)CCn3c2c4ccncc4,MAPK
Fc1ccc(cc1)c2nc([nH]c2c3ccncc3)c4ccccc4,MAPK
[NH3]Cc1ccc(cc1)c2nc(c([nH]2)c3ccncc3)c4ccccc4,MAPK
C(C[NH]1CCOCC1)Cc2ccc(cc2)c3nc(c([nH]3)c4ccncc4)c5ccccc5,MAPK
CCC(CC)Nc1nccc(n1)c2c(ncn2C3CC[NH](C)CC3)c4ccc(F)cc4,MAPK
CNc1nccc(n1)c2c(ncn2C3CCC4(CC3)OCCO4)c5ccc(F)cc5,MAPK
Fc1ccc(cc1)c2ncn(CCC[NH]3CCOCC3)c2c4ccnc5ccccc45,MAPK
Fc1ccc(cc1)c2ncn(CCC[NH]3CCOCC3)c2c4ccnc5ccccc45,MAPK
C[NH]1CCC(CC1)n2cnc(c2c3ccnc(NCc4ccccc4)n3)c5ccc(F)cc5,MAPK
CCC(CC)(CC)OC(=O)N1CCC(CC1)c2nc(c([nH]2)c3ccncn3)c4ccc(F)cc4,MAPK
CCc1nc(c(o1)c2cccc(OC)c2)c3ccncc3,MAPK
Fc1ccc(cc1)c2nc(oc2c3ccncc3)c4ccccc4,MAPK
CCS(=O)c1ccc(cc1)n2cc(c(n2)c3ccc(F)cc3)c4ccncc4,MAPK
CCC(CC)(CC)C1[NH2]C(C(=C1)c2ccncc2)c3ccc(F)cc3,MAPK
CC[NH]1CCC(CC1)C2[NH2]C(C(=C2)c3ccncc3)c4ccc(F)cc4,MAPK
COC1CCC(CC1)C2[NH2]C(C(=C2)c3ccncc3)c4ccc(F)cc4,MAPK
CCS(=C)c1ccc(cc1)c2nc(c([nH]2)c3ccncc3)c4ccc(F)cc4,MAPK
Nc1ccc2c(c([nH]c2n1)c3ccc(F)cc3)c4ccncc4,MAPK
CCc1ccc(cc1)c2cc(nn2c3ccc(cc3)C(=O)N)C(F)(F)F,MAPK
CCn1cc(c([nH]1)c2ccncc2)c3ccc(F)cc3,MAPK
Fc1ccc(cc1)c2cn[nH]c2c3ccnc(NC4CC[NH](Cc5ccccc5)CC4)c3,MAPK
CC(Nc1nccc(n1)c2c(nc(C3CC[NH2]CC3)n2C)c4cccc(c4)C(F)(F)F)c5ccccc5,MAPK
"""

stahl_df = pd.read_csv(StringIO(csv_data))

stahl_df.head()

,smiles,target
0,CC(=O)NC1C(OC(=CC1[NH+]=C(N)N)C(=O)[O-])C(=O)C...,neuraminidase
1,CCC(CC)C(=O)Nc1cc(ccc1NC(=O)C)C(=O)[O-],neuraminidase
2,NC(N)[NH+]=C1C=C(C=CC1N2C(=O)CCC2(CO)CO)C(=O)[O-],neuraminidase
3,CCC(CC)Nc1cc(ccc1N2C(=O)CCC2(CO)CO)C(=O)[O-],neuraminidase
4,CCCN(CCc1ccccc1)C(=O)C2OC(=CC([NH3+])C2NC(=O)C...,neuraminidase


Molecular descriptors are mathematical representations of chemical systems, obtained from
well‐defined algorithm. There are many different types of descriptors. They are commonly
classified into:
<br> 
- 2D descriptors (they can be calculated from connectivity table. Do not depend on
the conformation of the molecules and are used for the study of large groups of molecules)

- 3D descriptors (these descriptors are dependent on the internal coordinates of the molecules)
<br>

RDKit provides access to a large collection of molecular descriptors, including physicochemical, topological, constitutional, and fragment-based descriptors. In this practical session, we will calculate a comprehensive set of 1D and 2D molecular descriptors directly from the molecular structures represented as SMILES strings.

In [ ]:
#calculate the basic descriptors for each molecule in the dataframe. The descriptors to calculate are: MW, LogP, TPSA, HBA, HBD, RotBonds, Rings

def basic_desc(m):
 return {'MW':Descriptors.MolWt(m),'LogP':Descriptors.MolLogP(m),'TPSA':rdMolDescriptors.CalcTPSA(m),'HBA':rdMolDescriptors.CalcNumHBA(m),'HBD':rdMolDescriptors.CalcNumHBD(m),'RotBonds':rdMolDescriptors.CalcNumRotatableBonds(m),'Rings':rdMolDescriptors.CalcNumRings(m)}

desc=pd.DataFrame(stahl_df.mol.apply(basic_desc).tolist())

#join the descriptor dataframe with the original dataframe to have all the information in one dataframe
stahl_df=pd.concat([stahl_df,desc],axis=1)
stahl_df.head()

In [ ]:
stahl_df.describe()

In [ ]:
desc.hist(figsize=(10,8), bins=15); plt.tight_layout()

In [ ]:
# density plot for the logP descriptor depending on the target value (0 or 1) using seaborn
sns.displot(stahl_df, x="LogP", hue="target", kind="kde", fill=True, height=5, aspect=1.0)

In [ ]:
# boxplot for the MW descriptor depending on the target value (0 or 1), with different colors for each target using seaborn
sns.boxplot(data=stahl_df, x="target", y="LogP", hue="target", legend=True)


<div class="alert alert-block alert-warning">
<strong>Interpreting the LogP Distributions</strong><br>

Examine the boxplot comparing the LogP values of neuraminidase and MAPK ligands.
<br>
What does this observation suggest about the physicochemical requirements of ligands binding to these two targets?
</div>

### Full RDKit Descriptor Set

In the previous sections, we focused on a small set of physicochemical descriptors that are easy to interpret. However, chemoinformatics analyses often benefit from a more comprehensive description of molecular structure.

RDKit provides access to a large collection of molecular descriptors, including constitutional, topological, electronic, and physicochemical properties. In the next step, we will calculate all available RDKit descriptors for every molecule in the dataset and assemble them into a descriptor matrix.

Before using the descriptor matrix for multivariate analyses, several preprocessing steps are required:

1. **Remove invalid values** generated during descriptor calculation.
2. **Remove descriptors containing missing values** to ensure a complete dataset.
3. **Remove constant descriptors** (descriptors with the same value for all molecules), as they do not provide information useful for distinguishing compounds.

The resulting descriptor matrix will be used in the following sections to explore chemical space, calculate molecular similarity, and perform clustering analyses.

**Questions**

1. Why should descriptors containing missing values be removed before applying multivariate methods?
2. Why do constant descriptors provide no useful information for compound discrimination?

In [ ]:
funcs=Descriptors.descList
all_desc=pd.DataFrame([{n:(f(m) if True else np.nan) for n,f in funcs} for m in stahl_df.mol])
all_desc=all_desc.replace([np.inf,-np.inf],np.nan).dropna(axis=1)
all_desc=all_desc.loc[:,all_desc.nunique()>1]
all_desc.shape

## Principal Component Analysis (PCA)

The descriptor matrix generated in the previous section may contain hundreds of variables, making it difficult to visualize and interpret the relationships between molecules. 
<br>
**Principal Component Analysis (PCA)** is a dimensionality reduction technique that transforms the original descriptors into a smaller set of variables, called **principal components**, that capture most of the variability present in the dataset.

In this practical, PCA will allow us to visualize the distribution of compounds in chemical space, identify potential outliers, and investigate whether molecules naturally group according to their structural and physicochemical properties.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
X=StandardScaler().fit_transform(all_desc)

In [ ]:
pca=PCA(n_components=2)
scores=pca.fit_transform(X)
pca.explained_variance_ratio_

In [ ]:
plt.scatter(scores[:,0],scores[:,1])
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Chemical Space')

## Part 5. Molecular Similarity and Fingerprints

While molecular descriptors represent molecules using a set of numerical properties, **molecular fingerprints** encode structural information as binary vectors.

One of the available fingerprints in RDKit are the **Morgan fingerprints**. They correspond to circular fingerprints, generated by examining the atomic neighborhood around each atom up to a specified radius and encoding the resulting structural patterns into a fixed-length bit vector.

Once fingerprints have been calculated, molecular similarity can be quantified using the **Tanimoto coefficient**, the most commonly used similarity metric for binary fingerprints. The Tanimoto coefficient ranges from:

- **1.0**: identical fingerprints (maximum similarity)
- **0.0**: no common fingerprint features (minimum similarity)

In [ ]:
fps=[AllChem.GetMorganFingerprintAsBitVect(m,2,2048) for m in stahl_df.mol]

In [ ]:
if len(fps)>1:
 print(DataStructs.TanimotoSimilarity(fps[0],fps[1]))

In [ ]:
n=min(50,len(fps))
M=np.zeros((n,n))
for i in range(n):
 for j in range(n):
  M[i,j]=DataStructs.TanimotoSimilarity(fps[i],fps[j])
sns.heatmap(M,cmap='viridis')

## Part 6. Hierarchical Clustering

So far, we have represented each molecule as a point in chemical space using molecular descriptors and Principal Component Analysis (PCA). However, visual inspection alone may not be sufficient to identify groups of structurally or physicochemically similar compounds.

**Hierarchical clustering** is an unsupervised learning technique that groups molecules according to their similarity. Starting from individual compounds, the algorithm progressively merges the most similar objects, generating a tree-like structure known as a **dendrogram**.

In this exercise, we will apply **Ward's linkage method**, which seeks to minimize the variance within each cluster during the clustering process. As a result, compounds belonging to the same cluster are expected to share similar descriptor profiles and occupy nearby regions of chemical space.

The dendrogram provides a visual representation of the relationships between compounds and can help identify:

- Natural groups of similar molecules.
- Highly diverse or isolated compounds.
- Potential outliers.
- The appropriate number of clusters for subsequent analyses.

Once the dendrogram has been generated, it will be partitioned into **five clusters**, and each molecule will be assigned to one of them. 

These cluster assignments will be used in subsequent sections to compare molecular properties and select representative compounds from the chemical library.


In [ ]:
from scipy.cluster.hierarchy import linkage,dendrogram,cut_tree
Z=linkage(X,method='ward')

In [ ]:
plt.figure(figsize=(12,5))
dendrogram(Z)
plt.show()

In [ ]:
clusters=cut_tree(Z,n_clusters=5).flatten()+1
cluster_df=pd.concat([stahl_df[['smiles']],desc],axis=1)
cluster_df['Cluster']=clusters
cluster_df.head()

In [ ]:
cluster_df.Cluster.value_counts().sort_index()

## Part 7. Representative Compound Selection - Clustering

In [ ]:
#find the centroid of each cluster (using only the numeric descriptor columns).
desc_cols = list(desc.columns)

for c in sorted(cluster_df.Cluster.unique()):
    centroid = cluster_df.loc[cluster_df.Cluster==c, desc_cols].mean()

#find the molecule closest to the centroid of each cluster using the Euclidean distance,
#and store the index of the closest molecule in a list.
closest_molecules = []
for c in sorted(cluster_df.Cluster.unique()):
    members = cluster_df.loc[cluster_df.Cluster==c, desc_cols]
    centroid = members.mean()
    distances = np.linalg.norm(members.values - centroid.values, axis=1)
    closest_molecules.append(members.index[distances.argmin()])

# the representative molecule of each cluster is the one closest to its centroid
reps = closest_molecules
reps

#plot the chemical space with the clusters colored differently
plt.figure(figsize=(8,6))
for c in sorted(cluster_df.Cluster.unique()):
 mask = (cluster_df.Cluster==c).values
 plt.scatter(scores[mask,0],scores[mask,1],label=f'Cluster {c}')
for i, rep in enumerate(reps):
 plt.scatter(scores[rep,0],scores[rep,1],marker='x',s=200,c='black',label=f'Rep {i+1}')
#set the legend outside the plot
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()


In [ ]:
#Analysis of the results

#Compare LogP distributions across clusters and discuss physicochemical differences.

sns.boxplot(x=cluster_df['Cluster'],y=cluster_df['LogP'])

## Part 8. Representative Compound Selection - MaxMin

In [ ]:
from rdkit.SimDivFilters.rdSimDivPickers import MaxMinPicker
picker=MaxMinPicker()
picked=picker.LazyBitVectorPick(fps,len(fps),min(5,len(fps)))
picked

In [ ]:
# plot the chemical space with the picked molecules colored differently
plt.figure(figsize=(8,6))
for c in sorted(cluster_df.Cluster.unique()):
    mask = (cluster_df.Cluster==c).values
    plt.scatter(scores[mask,0],scores[mask,1],label=f'Cluster {c}')
for i, rep in enumerate(picked):
    plt.scatter(scores[rep,0],scores[rep,1],marker='x',s=200,c='black',label=f'Rep {i+1}')
#set the legend outside the plot
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()
